# 论文图：FCD 对比（带误差棒）/ Precision-Recall 权衡散点 / Coverage & Validity 分组柱状图

本 notebook 提供 3 张常用结果图的 **Matplotlib** 画图模板（可直接用于论文/补充材料）。

## 你需要准备什么数据？

- **FCD**：Baseline 与 Proposed 的均值与标准差（或标准误）
- **Precision-Recall**：每个模型的 (precision, recall) 点
- **Coverage & Validity**：每个模型的 coverage 与 validity（范围 0–1）

你可以：
- 在下方 `data = {...}` 里手动填写
- 或将数据整理为 CSV，再用 Pandas 读取（示例代码也给了）


In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

OUTDIR = Path('fig_out')
OUTDIR.mkdir(exist_ok=True)

# 统一图形参数（论文风格）
plt.rcParams.update({
    'figure.dpi': 160,
    'savefig.dpi': 300,
    'font.size': 11,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
    'legend.fontsize': 10,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
})

print('Output dir:', OUTDIR.resolve())


## 1) 填入你的数据（推荐）

把下面的数字替换成你的实验结果即可。


In [ ]:
data = {
    # FCD：越小越好
    'fcd': {
        'Baseline': {'mean': 0.85, 'err': 0.03},
        'Proposed': {'mean': 0.62, 'err': 0.02},
    },

    # Precision-Recall：越靠右上越理想
    # 你的模型用红色星号，其他模型用灰色圆/方
    'pr': [
        {'model': 'Baseline',   'precision': 0.74, 'recall': 0.58, 'marker': 'o'},
        {'model': 'Model-A',    'precision': 0.71, 'recall': 0.62, 'marker': 's'},
        {'model': 'Model-B',    'precision': 0.78, 'recall': 0.55, 'marker': 'D'},
        {'model': 'Proposed',   'precision': 0.83, 'recall': 0.69, 'marker': '*'},
    ],

    # Coverage & Validity：0~1 指标
    'cov_val': [
        {'model': 'Baseline', 'coverage': 0.61, 'validity': 0.92},
        {'model': 'Model-A',  'coverage': 0.66, 'validity': 0.90},
        {'model': 'Model-B',  'coverage': 0.70, 'validity': 0.88},
        {'model': 'Proposed', 'coverage': 0.78, 'validity': 0.94},
    ]
}

data


## （可选）2) 从 CSV 读取数据

如果你喜欢用文件驱动，把你的结果整理成以下格式即可：

### FCD（fcd.csv）
- columns: `model, mean, err`

### PR（pr.csv）
- columns: `model, precision, recall, marker`

### Coverage & Validity（cov_val.csv）
- columns: `model, coverage, validity`

然后取消注释读取部分。


In [ ]:
# fcd_df = pd.read_csv('fcd.csv')
# pr_df = pd.read_csv('pr.csv')
# cv_df = pd.read_csv('cov_val.csv')
#
# data = {
#   'fcd': {row['model']: {'mean': row['mean'], 'err': row['err']} for _, row in fcd_df.iterrows()},
#   'pr': pr_df.to_dict('records'),
#   'cov_val': cv_df.to_dict('records')
# }
#
# data


## 3) FCD 对比图（Bar Chart + Error Bars）

- Baseline：灰色柱
- Proposed：亮蓝色柱
- 带误差棒体现统计严谨性


In [ ]:
def plot_fcd_bar(fcd_dict, title='FCD (lower is better)', ylabel='FCD'):
    labels = list(fcd_dict.keys())
    means = [fcd_dict[k]['mean'] for k in labels]
    errs  = [fcd_dict[k]['err']  for k in labels]

    x = np.arange(len(labels))

    # 配色：按你的描述固定
    colors = []
    for lab in labels:
        if lab.lower() == 'baseline':
            colors.append('#9E9E9E')  # 灰
        elif lab.lower() == 'proposed':
            colors.append('#1E88E5')  # 亮蓝
        else:
            colors.append('#BDBDBD')

    fig, ax = plt.subplots(figsize=(4.2, 3.2))
    ax.bar(x, means, yerr=errs, capsize=4, color=colors, edgecolor='black', linewidth=0.7)

    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(axis='y', linestyle='--', linewidth=0.6, alpha=0.6)

    # 数值标注（可删）
    for i, (m, e) in enumerate(zip(means, errs)):
        ax.text(i, m + (e if e else 0) + 0.02*np.max(means), f'{m:.3f}', ha='center', va='bottom', fontsize=9)

    fig.tight_layout()
    return fig, ax

fig, ax = plot_fcd_bar(data['fcd'])
png = OUTDIR / 'fcd_bar.png'
pdf = OUTDIR / 'fcd_bar.pdf'
fig.savefig(png, bbox_inches='tight')
fig.savefig(pdf, bbox_inches='tight')
plt.show()
print('Saved:', png, 'and', pdf)


## 4) Precision-Recall 权衡图（Scatter Plot）

- 其他模型：灰色点（不同形状）
- Proposed：红色星号（视觉层级突出）
- 右上角为“理想区域”


In [ ]:
def plot_pr_scatter(pr_records, title='Precision–Recall Trade-off', xlabel='Recall', ylabel='Precision'):
    fig, ax = plt.subplots(figsize=(4.2, 3.6))

    # 先画非 Proposed
    for r in pr_records:
        name = r['model']
        p = float(r['precision'])
        rc = float(r['recall'])
        mk = r.get('marker', 'o')

        if name.lower() == 'proposed':
            continue

        ax.scatter(rc, p, marker=mk, s=70, c='#9E9E9E', edgecolors='black', linewidths=0.5, alpha=0.9, label=name)

    # 再画 Proposed
    for r in pr_records:
        if r['model'].lower() != 'proposed':
            continue
        p = float(r['precision'])
        rc = float(r['recall'])
        ax.scatter(rc, p, marker='*', s=180, c='#E53935', edgecolors='black', linewidths=0.6, label=r['model'], zorder=5)

    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.grid(True, linestyle='--', linewidth=0.6, alpha=0.6)

    # “理想区域”提示（可删）
    ax.text(0.72, 0.92, 'Ideal', ha='left', va='center', fontsize=10)

    # 去重 legend
    handles, labels = ax.get_legend_handles_labels()
    uniq = {}
    for h, l in zip(handles, labels):
        if l not in uniq:
            uniq[l] = h
    ax.legend(list(uniq.values()), list(uniq.keys()), loc='lower left', frameon=True)

    fig.tight_layout()
    return fig, ax

fig, ax = plot_pr_scatter(data['pr'])
png = OUTDIR / 'precision_recall_scatter.png'
pdf = OUTDIR / 'precision_recall_scatter.pdf'
fig.savefig(png, bbox_inches='tight')
fig.savefig(pdf, bbox_inches='tight')
plt.show()
print('Saved:', png, 'and', pdf)


## 5) Coverage & Validity 分组柱状图（Grouped Bar）

- 将两个 0–1 指标放在同一张图中对比
- 使用蓝绿冷色系，清晰且不刺眼


In [ ]:
def plot_grouped_cov_valid(cv_records, title='Coverage & Validity', ylabel='Score (0–1)'):
    df = pd.DataFrame(cv_records)
    models = df['model'].tolist()
    cov = df['coverage'].astype(float).values
    val = df['validity'].astype(float).values

    x = np.arange(len(models))
    width = 0.36

    fig, ax = plt.subplots(figsize=(6.2, 3.4))

    # 冷色系：蓝 / 绿
    ax.bar(x - width/2, cov, width, label='Coverage', color='#1E88E5', edgecolor='black', linewidth=0.6)
    ax.bar(x + width/2, val, width, label='Validity', color='#43A047', edgecolor='black', linewidth=0.6)

    ax.set_xticks(x)
    ax.set_xticklabels(models, rotation=0)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.set_ylim(0, 1.05)
    ax.grid(axis='y', linestyle='--', linewidth=0.6, alpha=0.6)
    ax.legend(loc='lower right', frameon=True)

    # 数值标注（可删）
    for i in range(len(models)):
        ax.text(x[i] - width/2, cov[i] + 0.02, f'{cov[i]:.2f}', ha='center', va='bottom', fontsize=9)
        ax.text(x[i] + width/2, val[i] + 0.02, f'{val[i]:.2f}', ha='center', va='bottom', fontsize=9)

    fig.tight_layout()
    return fig, ax

fig, ax = plot_grouped_cov_valid(data['cov_val'])
png = OUTDIR / 'coverage_validity_grouped.png'
pdf = OUTDIR / 'coverage_validity_grouped.pdf'
fig.savefig(png, bbox_inches='tight')
fig.savefig(pdf, bbox_inches='tight')
plt.show()
print('Saved:', png, 'and', pdf)


## 6) 输出文件

所有图片已保存到 `fig_out/`：

- `fcd_bar.png/pdf`
- `precision_recall_scatter.png/pdf`
- `coverage_validity_grouped.png/pdf`

投稿建议：在图注里明确误差棒含义（Std/SEM/CI），并说明 FCD 是越小越好。
